In [1]:
# S1 — JUST LOOK: What does the cascade produce at the t and b crossings?
#
# Stop thinking. Start looking.
# Top quark: gen3, up sector (a3=1, a5=2, a7=2) → ci=149
# Bottom quark: gen3, down sector (a3=1, a5=0, a7=2) → ci=191
# These are the SAME generation, different charge sectors.
# The cascade gives specific R_k values at each. What are they?

import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'scripts'))
os.environ['PYTHONUTF8'] = '1'
import numpy as np
from math import prod
from solenoid_system import SolenoidSystem
from solenoid_algebra import SA, RHO, OMEGA

primes = [2, 3, 5, 7]
p1, p2, p3, p4 = primes
P4 = prod(primes)
kappa = RHO
omega = OMEGA

sys0 = SolenoidSystem(primes=primes)
branches = sys0.all_branches()

# HIGH RESOLUTION: evaluate at EVERY integer from 1 to 210
# Not just coprime crossings — the full spatial structure
t_eval_full = np.arange(1, P4+1, dtype=float)
from solenoid_jax import warmup
warmup()
res_full = sys0.integrate_all_branches(branches, t_eval_full, float(P4+1), backend='jax')

# Also the standard coprime evaluation
cis = SA.coprime_indices(P4)
a3, a5, a7 = SA.sector_labels(cis)
sector_to_ci = {}
for idx in range(len(cis)):
    sector_to_ci[(a3[idx], a5[idx], a7[idx])] = cis[idx]

j4_vals = np.array([br[3] for br in branches])
j1_vals = np.array([br[0] for br in branches])
j2_vals = np.array([br[1] for br in branches])
j3_vals = np.array([br[2] for br in branches])

ci_t = sector_to_ci[(1, 2, 2)]  # top: gen3, up
ci_b = sector_to_ci[(1, 0, 2)]  # bottom: gen3, down
print(f'Top crossing: ci = {ci_t}')
print(f'Bottom crossing: ci = {ci_b}')
print(f'Delta ci = {ci_b - ci_t}')

# ═══ 1. RAW CASCADE DATA AT BOTH CROSSINGS ═══
print(f'\n{"=" * 70}')
print(f'1. RAW R_k VALUES AT TOP (ci={ci_t}) AND BOTTOM (ci={ci_b})')
print(f'{"=" * 70}')

def get_all_Rk(ci_val, res_dict, t_array):
    """Get R_k values at a specific ci for all branches and levels."""
    idx = np.where(t_array == ci_val)[0]
    if len(idx) == 0:
        return None
    idx = idx[0]
    data = np.zeros((len(branches), 4))
    for i, br in enumerate(branches):
        data[i] = res_dict[br][idx]
    return data

Rk_top = get_all_Rk(float(ci_t), res_full, t_eval_full)
Rk_bot = get_all_Rk(float(ci_b), res_full, t_eval_full)

# Wrapped versions
def wrap(x):
    w = np.mod(x, 2*np.pi)
    w[w > np.pi] -= 2*np.pi
    return w

Rk_top_w = np.column_stack([wrap(Rk_top[:,k]) for k in range(4)])
Rk_bot_w = np.column_stack([wrap(Rk_bot[:,k]) for k in range(4)])

for k in range(4):
    rms_t = np.sqrt(np.mean(Rk_top_w[:,k]**2))
    rms_b = np.sqrt(np.mean(Rk_bot_w[:,k]**2))
    mean_t = np.mean(Rk_top_w[:,k])
    mean_b = np.mean(Rk_bot_w[:,k])
    std_t = np.std(Rk_top_w[:,k])
    std_b = np.std(Rk_bot_w[:,k])
    ratio = rms_t / rms_b if rms_b > 1e-10 else float('inf')
    
    print(f'  R{k}: TOP rms={rms_t:.6f} mean={mean_t:+.4f} std={std_t:.4f}')
    print(f'       BOT rms={rms_b:.6f} mean={mean_b:+.4f} std={std_b:.4f}')
    print(f'       ratio TOP/BOT = {ratio:.6f}')

# ═══ 2. PER-SHEET (j4) PROFILES AT BOTH CROSSINGS ═══
print(f'\n{"=" * 70}')
print(f'2. PER-SHEET R3 PROFILES')
print(f'{"=" * 70}')

for label, Rk_w, ci in [('TOP', Rk_top_w, ci_t), ('BOT', Rk_bot_w, ci_b)]:
    print(f'\n  {label} (ci={ci}):')
    for j4 in range(7):
        mask = j4_vals == j4
        r3 = Rk_w[mask, 3]
        print(f'    j4={j4}: mean={np.mean(r3):+8.4f}, std={np.std(r3):.4f}, '
              f'n_wrap={np.sum(np.abs(Rk_top[mask,3] if label=="TOP" else Rk_bot[mask,3]) > np.pi)}')

# ═══ 3. THE FULL SPATIAL PROFILE: R_k(ci) for ci = 1 to 210 ═══
print(f'\n{"=" * 70}')
print(f'3. SPATIAL R3 RMS PROFILE (all ci from 1 to 210)')
print(f'{"=" * 70}')

rms_profile = np.zeros((len(t_eval_full), 4))
for ci_idx in range(len(t_eval_full)):
    for k in range(4):
        Rk = np.array([res_full[br][ci_idx, k] for br in branches])
        Rk_w = np.mod(Rk, 2*np.pi); Rk_w[Rk_w > np.pi] -= 2*np.pi
        rms_profile[ci_idx, k] = np.sqrt(np.mean(Rk_w**2))

# Show the profile around the t and b crossings
print(f'\n  R3 RMS profile near TOP (ci={ci_t}) and BOTTOM (ci={ci_b}):')
print(f'  {"ci":>4s}  {"R0":>8s} {"R1":>8s} {"R2":>8s} {"R3":>8s}  {"coprime?":>8s}')
for ci in range(ci_t-5, ci_b+6):
    if ci < 1 or ci > P4:
        continue
    idx = ci - 1  # t_eval_full starts at 1
    is_coprime = np.gcd(ci, P4) == 1
    marker = ''
    if ci == ci_t: marker = ' ← TOP'
    elif ci == ci_b: marker = ' ← BOT'
    print(f'  {ci:4d}  {rms_profile[idx,0]:8.4f} {rms_profile[idx,1]:8.4f} '
          f'{rms_profile[idx,2]:8.4f} {rms_profile[idx,3]:8.4f}  '
          f'{"YES" if is_coprime else "no":>8s}{marker}')

# ═══ 4. THE RATIO R(ci_t)/R(ci_b) AT EACH LEVEL ═══
print(f'\n{"=" * 70}')
print(f'4. RATIO OF RMS AT TOP/BOTTOM CROSSINGS')
print(f'{"=" * 70}')

idx_t = ci_t - 1
idx_b = ci_b - 1
for k in range(4):
    ratio = rms_profile[idx_t, k] / rms_profile[idx_b, k]
    print(f'  R{k}: TOP/BOT = {rms_profile[idx_t,k]:.6f} / {rms_profile[idx_b,k]:.6f} = {ratio:.6f}')

total_t = np.sqrt(np.sum(rms_profile[idx_t]**2))
total_b = np.sqrt(np.sum(rms_profile[idx_b]**2))
print(f'  Total: {total_t:.6f} / {total_b:.6f} = {total_t/total_b:.6f}')

# The RMS at each level should tell us about the VEV structure.
# If m_t/m_b is related to the cascade, it should appear in these ratios.

# ═══ 5. WHAT ABOUT THE SPATIAL DECAY BETWEEN ci_t AND ci_b? ═══
print(f'\n{"=" * 70}')
print(f'5. SPATIAL DECAY FROM ci={ci_t} TO ci={ci_b}')
print(f'{"=" * 70}')

delta_ci = ci_b - ci_t  # = 42!
print(f'  Delta ci = {delta_ci}')
print(f'  P4/p3 = {P4//p3}')
print(f'  THEY ARE THE SAME: Delta ci = P4/p3 = {delta_ci}!')

# The spatial decay exp(-kappa * delta_ci):
decay = np.exp(-kappa * delta_ci)
print(f'\n  exp(-kappa * delta_ci) = exp(-{kappa:.6f} * {delta_ci}) = {decay:.6f}')

# Actual RMS ratios:
for k in range(4):
    ratio = rms_profile[idx_b, k] / rms_profile[idx_t, k]
    expected = decay
    print(f'  R{k}: BOT/TOP = {ratio:.6f}, exp(-kappa*42) = {expected:.6f}, '
          f'actual/expected = {ratio/expected:.4f}')

# ═══ 6. THE KEY DISCOVERY ═══
print(f'\n{"=" * 70}')
print(f'6. THE KEY: Delta ci BETWEEN TOP AND BOTTOM IS P4/p3 = 42')
print(f'{"=" * 70}')

print(f'''
  Top quark (gen3, up):   ci = {ci_t}
  Bottom quark (gen3, down): ci = {ci_b}
  
  Delta ci = {ci_b} - {ci_t} = {delta_ci} = P4/p3 = {P4//p3}
  
  This is NOT a coincidence. The CRT assigns:
    UP gen3 (a5=2, a7=2) → ci = {ci_t}
    DOWN gen3 (a5=0, a7=2) → ci = {ci_b}
  
  The isospin step from UP to DOWN shifts ci by:
    Delta ci = ci(a5=0) - ci(a5=2) = {ci_b} - {ci_t} = {delta_ci}
  
  This spacing IS the mass ratio P4/p3 = 42!
  The t/b mass ratio is literally the SPATIAL DISTANCE between
  the top and bottom crossings on the solenoid.
  
  But the actual m_t/m_b is 41.25, not 42. The difference comes
  from how the cascade processes the signal between ci={ci_t} and ci={ci_b}.
  The exponential decay exp(-kappa*42) would give a specific factor.
  The wrapping and inner-level structure modify this.
''')

# ═══ 7. ALL GEN3 CROSSINGS: the full isospin step ═══
print(f'{"=" * 70}')
print(f'7. ALL GEN3 CROSSING POSITIONS (a7=2, all a5)')
print(f'{"=" * 70}')

for a5_val in range(4):
    ci = sector_to_ci.get((1, a5_val, 2))
    if ci is not None:
        print(f'  a5={a5_val}: ci={ci:4d}  (delta from a5=0: {ci - ci_b:+4d})')

# Check: is the step ALWAYS 42 between a5=0 and a5=2?
print(f'\n  Isospin step for ALL generations:')
for a7_val in range(6):
    ci_down = sector_to_ci.get((1, 0, a7_val))
    ci_up = sector_to_ci.get((1, 2, a7_val))
    if ci_down and ci_up:
        delta = ci_up - ci_down
        print(f'  a7={a7_val}: ci_down={ci_down:3d}, ci_up={ci_up:3d}, '
              f'delta={delta:+4d} (mod 210: {delta % P4})')

# ═══ 8. THE CASCADE BETWEEN ci_t AND ci_b ═══
print(f'\n{"=" * 70}')
print(f'8. WHAT HAPPENS IN THE CASCADE BETWEEN ci={ci_t} AND ci={ci_b}')
print(f'{"=" * 70}')

# Show the R3 RMS at every integer ci between top and bottom
print(f'  ci  R3_rms    decay_from_top  actual/decay')
rms_t_r3 = rms_profile[idx_t, 3]
for ci in range(ci_t, ci_b+1):
    idx = ci - 1
    expected = rms_t_r3 * np.exp(-kappa * (ci - ci_t))
    ratio = rms_profile[idx, 3] / expected if expected > 1e-10 else 0
    marker = ''
    if ci == ci_t: marker = ' ← TOP'
    elif ci == ci_b: marker = ' ← BOT'
    elif np.gcd(ci, P4) == 1: marker = ' (coprime)'
    print(f'  {ci:3d}  {rms_profile[idx,3]:.6f}  {expected:.6f}  {ratio:.4f}{marker}')

  JAX [CPU (1 device(s))]: 210 branches, 210 eval pts, T=211.0 — 1.70s
Top crossing: ci = 149
Bottom crossing: ci = 191
Delta ci = 42

1. RAW R_k VALUES AT TOP (ci=149) AND BOTTOM (ci=191)
  R0: TOP rms=0.010874 mean=-0.0109 std=0.0001
       BOT rms=0.010975 mean=-0.0110 std=0.0000
       ratio TOP/BOT = 0.990754
  R1: TOP rms=0.028222 mean=+0.0282 std=0.0006
       BOT rms=0.027498 mean=+0.0275 std=0.0000
       ratio TOP/BOT = 1.026360
  R2: TOP rms=0.041718 mean=-0.0417 std=0.0012
       BOT rms=0.043644 mean=-0.0436 std=0.0001
       ratio TOP/BOT = 0.955875
  R3: TOP rms=0.299519 mean=-0.2995 std=0.0012
       BOT rms=0.279486 mean=+0.2795 std=0.0001
       ratio TOP/BOT = 1.071676

2. PER-SHEET R3 PROFILES

  TOP (ci=149):
    j4=0: mean= -0.3002, std=0.0011, n_wrap=0
    j4=1: mean= -0.2999, std=0.0011, n_wrap=0
    j4=2: mean= -0.2997, std=0.0011, n_wrap=0
    j4=3: mean= -0.2995, std=0.0011, n_wrap=0
    j4=4: mean= -0.2993, std=0.0011, n_wrap=0
    j4=5: mean= -0.2991, std=0

In [2]:
# S2 — THE R3 OSCILLATION: What creates the ~14-period cycle?
#
# Between ci=149 and ci=191, R3 oscillates with period ~14.
# 14 = p1*p4 = 2*7. Is this the R3 driving frequency?
# The R3 ODE has driving from omega*sin(theta3) where theta3 involves p4=7.
# The effective period at R3 should be related to p4.
#
# Also: the TOP and BOT crossings both sit at approximately the SAME PHASE
# in the oscillation. R3(149) = 0.2995, R3(191) = 0.2795.
# The ratio 0.2795/0.2995 = 0.933. What determines this?

import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'scripts'))
os.environ['PYTHONUTF8'] = '1'
import numpy as np
from math import prod, gcd
from solenoid_system import SolenoidSystem
from solenoid_algebra import SA, RHO, OMEGA

primes = [2, 3, 5, 7]
p1, p2, p3, p4 = primes
P4 = prod(primes)
kappa = RHO
omega = OMEGA

sys0 = SolenoidSystem(primes=primes)
branches = sys0.all_branches()

# Use the full-resolution data from S1
t_eval_full = np.arange(1, P4+1, dtype=float)
from solenoid_jax import warmup
warmup()
res_full = sys0.integrate_all_branches(branches, t_eval_full, float(P4+1), backend='jax')

# ═══ 1. R3 OSCILLATION PERIOD ═══
print('=' * 70)
print('1. R3 OSCILLATION: Period and Phase')
print('=' * 70)

# Compute R3 RMS at every ci
rms_r3 = np.zeros(P4)
for ci_idx in range(P4):
    Rk = np.array([res_full[br][ci_idx, 3] for br in branches])
    Rk_w = np.mod(Rk, 2*np.pi); Rk_w[Rk_w > np.pi] -= 2*np.pi
    rms_r3[ci_idx] = np.sqrt(np.mean(Rk_w**2))

# Find zeros (minima) in the R3 profile
# The zeros are where the cascade passes through zero
zeros = []
for i in range(1, P4-1):
    if rms_r3[i] < rms_r3[i-1] and rms_r3[i] < rms_r3[i+1] and rms_r3[i] < 0.05:
        zeros.append(i+1)  # ci = index + 1

print(f'  R3 zeros (minima < 0.05) at ci:')
print(f'  {zeros}')
if len(zeros) > 1:
    spacings = np.diff(zeros)
    print(f'  Spacings between zeros: {spacings}')
    print(f'  Mean spacing: {np.mean(spacings):.1f}')
    print(f'  Expected P4/p4 = {P4//p4} = {P4/p4:.1f}')
    # Actually the driving is at omega = 2*pi per period.
    # The R3 level has p4 = 7 sheets. The effective period should be P4/p4 = 30.
    # But the oscillation period is ~14. Why?
    # 14 = P4/(p3*p4/gcd(p3,p4)) ... no.
    # 14 = 2*7 = p1*p4. Hmm.
    # Actually: the RMS oscillation depends on the MEAN wrapped R3.
    # The mean R3 ∝ sin(omega*ci/P4) at the j4=0 level (or some combination).
    # The period of sin(omega*t/P4) when evaluated at integer t:
    # period = P4/gcd(P4, round_to_nearest_integer(omega/(2*pi)*P4))
    # omega = 2*pi, so omega*t = 2*pi*t, and sin(2*pi*t/P4) has period P4.
    # But we're looking at the wrapped RMS, which depends on how many sheets
    # are in phase.

# ═══ 2. THE STEADY-STATE STRUCTURE ═══
print(f'\n{"=" * 70}')
print('2. THE STEADY-STATE R3: What determines the RMS in the far field?')
print('=' * 70)

# At large ci (far from wrapping zone), R3 is in steady state.
# The steady-state R3 for each branch is:
# R3_ss(j4) = (omega*j4/p4) * (1/(omega^2 + kappa^2)) * kappa ≈ ...
# Actually, the cascade ODE at level 3 is:
# dR3/dt = epsilon*sin(theta3) - kappa*R3 + (kappa/p4)*R2
# In steady state: 0 = epsilon*sin(theta3_ss) - kappa*R3_ss + (kappa/p4)*R2_ss
# This gives R3_ss that depends on theta3_ss (which depends on j4 and t).

# At ci far from the wrapping zone, R3 is small (|R3| << pi).
# In this regime, sin(theta3) ≈ theta3 = omega*t - j4*R3.
# Wait, theta_k = omega*t - j_{k+1}*R_k for the coupling...
# I need to be more careful about what drives R3.

# Let me just measure the ACTUAL steady-state R3 per branch.
# At ci=149 (TOP) and ci=191 (BOT):
ci_t, ci_b = 149, 191
idx_t, idx_b = ci_t-1, ci_b-1

# The R3 values per branch
R3_t = np.array([res_full[br][idx_t, 3] for br in branches])
R3_b = np.array([res_full[br][idx_b, 3] for br in branches])

# The mean and std
print(f'  R3 at TOP (ci={ci_t}): mean={np.mean(R3_t):+.6f}, std={np.std(R3_t):.6f}')
print(f'  R3 at BOT (ci={ci_b}): mean={np.mean(R3_b):+.6f}, std={np.std(R3_b):.6f}')
print(f'  |mean ratio| = {abs(np.mean(R3_t)/np.mean(R3_b)):.6f}')

# The R3 values at these crossings are dominated by the MEAN (std << mean).
# The mean is the steady-state value, which depends on ci through the driving.

# ═══ 3. HOW DOES THE MEAN R3 DEPEND ON ci? ═══
print(f'\n{"=" * 70}')
print('3. MEAN R3 vs ci: The driving phase')
print('=' * 70)

mean_r3 = np.zeros(P4)
for ci_idx in range(P4):
    R3 = np.array([res_full[br][ci_idx, 3] for br in branches])
    mean_r3[ci_idx] = np.mean(R3)

# The mean R3 is the j4=0 steady-state value (since j4 only adds a linear slope).
# Actually, it's the mean across all j4 sheets.
# For a linear profile R3 = a + b*j4, the mean = a + b*3 (mean of j4=0..6 is 3).

# Show mean R3 from ci=140 to ci=200
print(f'  ci  mean_R3     |mean_R3|   sin(omega*ci)  R3/sin')
for ci in range(140, 201):
    idx = ci - 1
    s = np.sin(omega * ci)
    ratio = mean_r3[idx] / s if abs(s) > 0.001 else float('nan')
    marker = ''
    if ci == 149: marker = ' ← TOP'
    elif ci == 191: marker = ' ← BOT'
    print(f'  {ci:3d}  {mean_r3[idx]:+10.6f}  {abs(mean_r3[idx]):10.6f}  '
          f'{s:+10.6f}  {ratio:+10.4f}{marker}')

# ═══ 4. IS THERE A PHASE RELATIONSHIP? ═══
print(f'\n{"=" * 70}')
print('4. R3 vs sin(omega*ci): Is R3 proportional to the driving?')
print('=' * 70)

# For the steady state, R3_ss ∝ sin(omega*ci + phase) with some amplitude.
# The driving in the cascade is ε*sin(theta_k) where theta_k involves omega*t.
# At level 3: theta3 = omega*t - j4*R3 (approximately, in linearized regime).
# For small R3: theta3 ≈ omega*t, so R3 ∝ sin(omega*t).
# But omega = 2*pi and t = ci (integer), so sin(omega*ci) = sin(2*pi*ci) = 0 always!

# Wait — sin(2*pi*ci) = 0 for all integer ci. That means the DIRECT driving
# at integer crossings is ZERO. The R3 value comes from the ACCUMULATED
# driving between crossings.

# Actually, the cascade is integrated CONTINUOUSLY from t=0 to t=T.
# The crossings at integer ci are EVALUATION points, not driving points.
# Between ci and ci+1, the driving sin(omega*t) = sin(2*pi*t) oscillates
# through one complete cycle. The R3 at ci is the result of all this.

# So R3(ci) ≈ steady-state amplitude × sin(phi(ci)) where phi depends
# on the phase accumulated from the initial condition and the driving.

# The fact that R3 oscillates with period ~14 means there's a beat
# frequency between the driving (period 1 in t) and some other frequency.

# What frequency could give period 14?
# If R3 has a natural frequency f_nat, the beat is |f_drive - f_nat|.
# f_drive = omega/(2*pi) = 1 (per unit t)
# Beat period = 1/|1 - f_nat|
# For period 14: |1 - f_nat| = 1/14, so f_nat = 13/14 or 15/14.

# OR: the period comes from the discrete evaluation at integer points.
# R3 at integer ci samples the continuous R3(t) at t=ci.
# If R3(t) has a SLOW modulation, sampling at integers aliases it.

# The continuous R3(t) in steady state has the driving frequency omega = 2*pi.
# At integer t: sin(2*pi*ci) = 0. The residual comes from:
# 1. The transient (decaying with exp(-kappa*t))
# 2. The inner-level coupling (from R2, which has its own frequency structure)

# Let me check: what is R2 doing?
mean_r2 = np.zeros(P4)
for ci_idx in range(P4):
    R2 = np.array([res_full[br][ci_idx, 2] for br in branches])
    mean_r2[ci_idx] = np.mean(R2)

# The R2 drives R3 through the coupling term (kappa/p4)*R2.
# Show R2 and R3 together:
print(f'  ci  mean_R2      mean_R3     ratio R3/R2')
for ci in range(149, 192, 3):
    idx = ci - 1
    r = mean_r3[idx]/mean_r2[idx] if abs(mean_r2[idx]) > 1e-6 else 0
    marker = ''
    if ci == 149: marker = ' ← TOP'
    elif ci == 191: marker = ' ← BOT'
    print(f'  {ci:3d}  {mean_r2[idx]:+10.6f}  {mean_r3[idx]:+10.6f}  {r:+10.4f}{marker}')

# ═══ 5. THE FUNDAMENTAL: Each level's contribution to R3 ═══
print(f'\n{"=" * 70}')
print('5. LEVEL-BY-LEVEL MEANS AT TOP AND BOTTOM')
print('=' * 70)

for label, ci in [('TOP', 149), ('BOT', 191)]:
    idx = ci - 1
    print(f'\n  {label} (ci={ci}):')
    for k in range(4):
        Rk = np.array([res_full[br][idx, k] for br in branches])
        print(f'    R{k}: mean={np.mean(Rk):+10.6f}, std={np.std(Rk):.6f}, '
              f'|mean|/std={abs(np.mean(Rk))/np.std(Rk):.1f}')

# The means dominate over std (ratio >> 1) at these crossings.
# So the R3 value is essentially the MEAN across all 210 branches.
# This mean is the steady-state cascade response.

# ═══ 6. WHAT CREATES THE R3 AT COPRIME CROSSINGS? ═══
print(f'\n{"=" * 70}')
print('6. THE R3 SIGNAL AT COPRIME CROSSINGS IS THE INNER-LEVEL CASCADE')
print('=' * 70)

# At integer t, sin(omega*t) = sin(2*pi*t) = 0.
# So the DIRECT R3 driving is zero at every crossing.
# The R3 value is entirely from the INNER-LEVEL coupling (kappa/p4)*R2.
# R2 in turn gets from (kappa/p3)*R1, and R1 from (kappa/p2)*R0.
# R0 gets from the direct driving: epsilon*sin(omega*t).
# But at integer t: sin(omega*t) = 0. So R0 driving is also zero!

# This means: at integer crossings, ALL R_k values are from the
# ACCUMULATED response between crossings, not from the instantaneous driving.
# The cascade is a filter that integrates the driving over each period
# and produces a residual at the integer evaluation points.

# The residual depends on the PHASE OFFSET between the driving and
# the natural response at each level. This phase offset varies slowly
# with ci, creating the observed oscillation.

# For R3, the oscillation period ≈ 14 = 2*7 = p1*p4.
# For R2, check the period:
zeros_r2 = []
for i in range(50, P4-1):
    if abs(mean_r2[i]) < abs(mean_r2[i-1]) and abs(mean_r2[i]) < abs(mean_r2[i+1]) and abs(mean_r2[i]) < 0.002:
        zeros_r2.append(i+1)

if len(zeros_r2) > 1:
    spacings_r2 = np.diff(zeros_r2)
    print(f'  R2 zero crossings at ci: {zeros_r2[:20]}')
    print(f'  R2 spacings: {spacings_r2[:15]}')
    print(f'  R2 mean period: {np.mean(spacings_r2):.1f}')

# For R1:
mean_r1 = np.zeros(P4)
for ci_idx in range(P4):
    R1 = np.array([res_full[br][ci_idx, 1] for br in branches])
    mean_r1[ci_idx] = np.mean(R1)

zeros_r1 = []
for i in range(50, P4-1):
    if abs(mean_r1[i]) < abs(mean_r1[i-1]) and abs(mean_r1[i]) < abs(mean_r1[i+1]) and abs(mean_r1[i]) < 0.002:
        zeros_r1.append(i+1)

if len(zeros_r1) > 1:
    spacings_r1 = np.diff(zeros_r1)
    print(f'  R1 zero crossings at ci: {zeros_r1[:20]}')
    print(f'  R1 spacings: {spacings_r1[:15]}')
    print(f'  R1 mean period: {np.mean(spacings_r1):.1f}')

# R3 period ~14 = 2*7 = p1*p4
# R2 period = ? (should be p1*p3 = 10?)
# R1 period = ? (should be p1*p2 = 6?)

print(f'\n  Expected periods from prime structure:')
print(f'  R3: p1*p4 = {p1*p4} = 14?')
print(f'  R2: p1*p3 = {p1*p3} = 10?')
print(f'  R1: p1*p2 = {p1*p2} = 6?')

  JAX [CPU (1 device(s))]: 210 branches, 210 eval pts, T=211.0 — 1.59s
1. R3 OSCILLATION: Period and Phase
  R3 zeros (minima < 0.05) at ci:
  [95, 111, 126, 141, 156, 171, 186, 201]
  Spacings between zeros: [16 15 15 15 15 15 15]
  Mean spacing: 15.1
  Expected P4/p4 = 30 = 30.0

2. THE STEADY-STATE R3: What determines the RMS in the far field?
  R3 at TOP (ci=149): mean=-0.299516, std=0.001162
  R3 at BOT (ci=191): mean=+0.279486, std=0.000111
  |mean ratio| = 1.071668

3. MEAN R3 vs ci: The driving phase
  ci  mean_R3     |mean_R3|   sin(omega*ci)  R3/sin
  140   +0.060975    0.060975   -0.000000        +nan
  141   -0.009727    0.009727   -0.000000        +nan
  142   -0.067168    0.067168   +0.000000        +nan
  143   -0.116570    0.116570   -0.000000        +nan
  144   -0.168436    0.168436   -0.000000        +nan
  145   -0.225323    0.225323   -0.000000        +nan
  146   -0.276680    0.276680   -0.000000        +nan
  147   -0.307613    0.307613   +0.000000        +nan
  

In [3]:
# S3 — SETUP: Full-resolution cascade + precompute profiles
#
# This cell does the expensive computation once. All subsequent cells use it.

import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'scripts'))
os.environ['PYTHONUTF8'] = '1'
import numpy as np
from math import prod, gcd
from solenoid_system import SolenoidSystem
from solenoid_algebra import SA, RHO, OMEGA, CP_PAIRS

primes = [2, 3, 5, 7]
p1, p2, p3, p4 = primes
P4 = prod(primes)
kappa = RHO; omega = OMEGA

sys0 = SolenoidSystem(primes=primes)
branches = sys0.all_branches()
t_eval = np.arange(1, P4+1, dtype=float)
from solenoid_jax import warmup
warmup()
res = sys0.integrate_all_branches(branches, t_eval, float(P4+1), backend='jax')

cis = SA.coprime_indices(P4)
a3, a5, a7 = SA.sector_labels(cis)
sector_to_ci = {}
for idx in range(len(cis)):
    sector_to_ci[(a3[idx], a5[idx], a7[idx])] = cis[idx]

j_vals = np.array([[br[k] for k in range(4)] for br in branches])
gen_map = {0:1, 3:1, 1:2, 4:2, 2:3, 5:3}

def wrap(x):
    w = np.mod(x, 2*np.pi)
    w[w > np.pi] -= 2*np.pi
    return w

# Precompute profiles at every ci
print('Precomputing profiles at all 210 positions...')
N = len(t_eval)
profiles = {}
for key in ['mean','rms','std','mean_w','rms_w','std_w']:
    profiles[key] = np.zeros((N, 4))
profiles['energy_w'] = np.zeros(N)
profiles['wrap_frac'] = np.zeros((N, 4))

for ci_idx in range(N):
    for k in range(4):
        Rk = np.array([res[br][ci_idx, k] for br in branches])
        Rk_w = wrap(Rk)
        profiles['mean'][ci_idx, k] = np.mean(Rk)
        profiles['rms'][ci_idx, k] = np.sqrt(np.mean(Rk**2))
        profiles['std'][ci_idx, k] = np.std(Rk)
        profiles['mean_w'][ci_idx, k] = np.mean(Rk_w)
        profiles['rms_w'][ci_idx, k] = np.sqrt(np.mean(Rk_w**2))
        profiles['std_w'][ci_idx, k] = np.std(Rk_w)
        profiles['wrap_frac'][ci_idx, k] = np.sum(np.abs(Rk) > np.pi) / len(branches)
    profiles['energy_w'][ci_idx] = np.sum(profiles['rms_w'][ci_idx]**2)

print(f'Done. {N} positions, {len(branches)} branches, 4 levels.')

  JAX [CPU (1 device(s))]: 210 branches, 210 eval pts, T=211.0 — 1.59s
Precomputing profiles at all 210 positions...
Done. 210 positions, 210 branches, 4 levels.

In [4]:
# A1: COMPLETE WRAPPING MAP — all 48 coprime crossings + wrapping zone non-coprime
#
# Show wrapped RMS at all 4 levels, wrapping fraction, and sector labels.
# This is the fundamental data table of the solenoid.

print('A1: COMPLETE SPATIAL PROFILE')
print('=' * 90)
print(f'  {"ci":>4s} {"cop":>3s}  {"R0":>8s} {"R1":>8s} {"R2":>8s} {"R3":>8s}  '
      f'{"E_w":>8s} {"wR3":>5s}  sector')
print(f'  {"-"*88}')

for ci_idx in range(N):
    ci = int(t_eval[ci_idx])
    is_cop = gcd(ci, P4) == 1
    # Show: all coprime + non-coprime in wrapping zone (ci < 55)
    if not is_cop and ci > 55:
        continue
    sector = ''
    for idx2 in range(len(cis)):
        if cis[idx2] == ci:
            s_a3 = 'Q' if a3[idx2] == 1 else 'L'
            s_a5 = ['d','u1','u','d3'][a5[idx2]]
            gen = gen_map.get(a7[idx2], '?')
            sector = f'{s_a3} {s_a5} g{gen}'
            break
    wf = profiles['wrap_frac'][ci_idx, 3]
    wf_str = f'{wf*100:4.0f}%' if wf > 0.005 else '   .'
    print(f'  {ci:4d} {"Y" if is_cop else ".":>3s}  '
          f'{profiles["rms_w"][ci_idx,0]:8.5f} {profiles["rms_w"][ci_idx,1]:8.5f} '
          f'{profiles["rms_w"][ci_idx,2]:8.5f} {profiles["rms_w"][ci_idx,3]:8.5f}  '
          f'{profiles["energy_w"][ci_idx]:8.5f} {wf_str:>5s}  {sector}')

A1: COMPLETE SPATIAL PROFILE
    ci cop        R0       R1       R2       R3       E_w   wR3  sector
  ----------------------------------------------------------------------------------------
     1   Y   0.29677  0.47084  0.92137  1.38023   3.06371   86%  L d g1
     2   .   0.57375  0.92679  1.79101  1.87836   7.92407   86%  
     3   .   0.83226  1.34003  1.85917  1.62378   8.58153   86%  
     4   .   1.07353  1.74645  1.66658  1.65979   9.73494   86%  
     5   .   1.29871  1.85243  1.57670  1.80700  10.86938   86%  
     6   .   1.50888  1.79307  1.62854  1.69787  11.02671   86%  
     7   .   1.70504  1.66512  1.78581  1.62476  11.50875   86%  
     8   .   1.88812  1.61254  1.77833  1.80070  12.57026   86%  
     9   .   2.05898  1.59842  1.65752  1.66445  12.31211   86%  
    10   .   2.21846  1.65500  1.70946  1.64913  13.30245   86%  
    11   Y   2.07559  1.61860  1.73710  1.84649  13.35501   86%  Q d g2
    12   .   1.93667  1.59845  1.74041  1.81028  12.61190   86%  
    

In [5]:
# A2: FOURIER ANALYSIS — spatial frequencies at each level
#
# What are the dominant spatial frequencies? How do they relate to the primes?

print('A2: FOURIER ANALYSIS OF SPATIAL PROFILES')
print('=' * 80)

for k in range(4):
    # Use wrapped mean as the signal
    signal = profiles['mean_w'][:, k]
    fft = np.fft.fft(signal)
    freqs = np.fft.fftfreq(N, d=1)
    power = np.abs(fft)**2
    power[0] = 0  # remove DC
    total_power = np.sum(power)
    
    # Top 8 frequencies
    top_idx = np.argsort(power)[::-1][:8]
    
    print(f'\n  R{k} (p={primes[k]}) — dominant spatial frequencies:')
    print(f'    {"freq":>8s} {"period":>8s} {"amplitude":>10s} {"power%":>8s}  interpretation')
    for fi in top_idx:
        f = abs(freqs[fi])
        period = 1/f if f > 1e-6 else float('inf')
        amp = np.abs(fft[fi]) / N
        pct = power[fi] / total_power * 100
        
        # Try to identify the period
        interp = ''
        if abs(period - P4) < 1: interp = f'= P4 = {P4}'
        for pa in primes:
            for pb in primes:
                if pa <= pb:
                    if abs(period - pa*pb) < 0.5: interp = f'= {pa}*{pb}'
                    if abs(period - P4/(pa*pb)) < 0.5: interp = f'= P4/({pa}*{pb})'
                    if abs(period - pa) < 0.5: interp = f'= p={pa}'
        for pa in primes:
            if abs(period - P4/pa) < 0.5: interp = f'= P4/{pa}'
        
        if pct > 0.5:  # only show significant
            print(f'    {f:8.4f} {period:8.1f} {amp:10.6f} {pct:7.1f}%  {interp}')
    
    # Also: what fraction of power is in the top 3?
    top3_power = sum(power[top_idx[:3]]) / total_power * 100
    print(f'    Top 3 contain {top3_power:.1f}% of total power')

# Also do RMS (not mean) to see if wrapping changes the spectrum
print(f'\n  --- WRAPPED RMS SPECTRUM (different from mean!) ---')
for k in [3]:  # just R3 for comparison
    signal = profiles['rms_w'][:, k]
    fft = np.fft.fft(signal)
    freqs = np.fft.fftfreq(N, d=1)
    power = np.abs(fft)**2
    dc = power[0]
    power[0] = 0
    total_power = np.sum(power)
    top_idx = np.argsort(power)[::-1][:5]
    
    print(f'\n  R{k} RMS — dominant frequencies:')
    print(f'    DC component: {np.abs(fft[0])/N:.6f} (mean RMS = {np.mean(signal):.6f})')
    for fi in top_idx:
        f = abs(freqs[fi])
        period = 1/f if f > 1e-6 else float('inf')
        amp = np.abs(fft[fi]) / N
        pct = power[fi] / total_power * 100
        if pct > 0.5:
            print(f'    freq={f:.4f}, period={period:.1f}, amp={amp:.6f}, power={pct:.1f}%')

A2: FOURIER ANALYSIS OF SPATIAL PROFILES

  R0 (p=2) — dominant spatial frequencies:
        freq   period  amplitude   power%  interpretation
      0.0143     70.0   0.065156     3.3%  = P4/3
      0.0143     70.0   0.065156     3.3%  = P4/3
      0.0190     52.5   0.065080     3.3%  = P4/(2*2)
      0.0190     52.5   0.065080     3.3%  = P4/(2*2)
      0.0238     42.0   0.064227     3.2%  = P4/5
      0.0238     42.0   0.064227     3.2%  = P4/5
      0.0095    105.0   0.064128     3.2%  = P4/2
      0.0095    105.0   0.064128     3.2%  = P4/2
    Top 3 contain 9.9% of total power

  R1 (p=3) — dominant spatial frequencies:
        freq   period  amplitude   power%  interpretation
      0.0048    210.0   0.121983    12.6%  = P4 = 210
      0.0048    210.0   0.121983    12.6%  = P4 = 210
      0.0095    105.0   0.108676    10.0%  = P4/2
      0.0095    105.0   0.108676    10.0%  = P4/2
      0.0143     70.0   0.093312     7.4%  = P4/3
      0.0143     70.0   0.093312     7.4%  = P4/3
 

In [6]:
# A3: GEN3 CHARGE QUARTET — all 4 charge sectors at a7=2 (gen3)
#
# Gen3 quarks at all 4 charge values: a5=0(d), a5=1, a5=2(u), a5=3
# What does the cascade produce at each? How do they compare?

print('A3: GEN3 (a7=2) COMPLETE CHARGE QUARTET')
print('=' * 80)

gen3_data = []
for a5_val in range(4):
    ci = sector_to_ci.get((1, a5_val, 2))
    if ci is None: continue
    ci_idx = ci - 1
    charge_names = {0: 'DOWN (d,s,b)', 1: 'iso-up-1', 2: 'UP (u,c,t)', 3: 'iso-down-3'}
    
    print(f'\n  a5={a5_val} [{charge_names[a5_val]}], ci={ci:3d}:')
    for k in range(4):
        print(f'    R{k}: mean={profiles["mean"][ci_idx,k]:+10.6f}  '
              f'mean_w={profiles["mean_w"][ci_idx,k]:+10.6f}  '
              f'rms_w={profiles["rms_w"][ci_idx,k]:8.6f}  '
              f'std_w={profiles["std_w"][ci_idx,k]:8.6f}  '
              f'wrap={profiles["wrap_frac"][ci_idx,k]:5.1%}')
    print(f'    energy_w = {profiles["energy_w"][ci_idx]:.6f}')
    gen3_data.append((a5_val, ci, profiles['rms_w'][ci_idx].copy(), profiles['energy_w'][ci_idx]))

# Pairwise ratios
print(f'\n  Pairwise RMS ratios between charge sectors:')
print(f'  {"pair":>12s}  {"R0":>8s} {"R1":>8s} {"R2":>8s} {"R3":>8s}  {"E ratio":>8s}')
for i in range(len(gen3_data)):
    for j in range(i+1, len(gen3_data)):
        a5_i, ci_i, rms_i, E_i = gen3_data[i]
        a5_j, ci_j, rms_j, E_j = gen3_data[j]
        ratios = rms_i / rms_j
        pair = f'a5={a5_i}/{a5_j}'
        print(f'  {pair:>12s}  {ratios[0]:8.4f} {ratios[1]:8.4f} {ratios[2]:8.4f} {ratios[3]:8.4f}  {E_i/E_j:8.4f}')

# The UP/DOWN ratio at gen3 should be related to m_t/m_b somehow
a5_0_data = [d for d in gen3_data if d[0] == 0][0]
a5_2_data = [d for d in gen3_data if d[0] == 2][0]
print(f'\n  KEY: UP(a5=2)/DOWN(a5=0) at gen3:')
for k in range(4):
    r = a5_2_data[2][k] / a5_0_data[2][k]
    print(f'    R{k}: {r:.6f}')
print(f'    Energy: {a5_2_data[3]/a5_0_data[3]:.6f}')
print(f'    Total RMS: {np.sqrt(a5_2_data[3])/np.sqrt(a5_0_data[3]):.6f}')
print(f'    PDG m_t/m_b = {172.69/4.183:.4f}')

A3: GEN3 (a7=2) COMPLETE CHARGE QUARTET

  a5=0 [DOWN (d,s,b)], ci=191:
    R0: mean= -0.010975  mean_w= -0.010975  rms_w=0.010975  std_w=0.000006  wrap= 0.0%
    R1: mean= +0.027498  mean_w= +0.027498  rms_w=0.027498  std_w=0.000040  wrap= 0.0%
    R2: mean= -0.043644  mean_w= -0.043644  rms_w=0.043644  std_w=0.000096  wrap= 0.0%
    R3: mean= +0.279486  mean_w= +0.279486  rms_w=0.279486  std_w=0.000111  wrap= 0.0%
    energy_w = 0.080894

  a5=1 [iso-up-1], ci=107:
    R0: mean= -0.009023  mean_w= -0.009023  rms_w=0.009231  std_w=0.001952  wrap= 0.0%
    R1: mean= +0.038577  mean_w= +0.038577  rms_w=0.039371  std_w=0.007866  wrap= 0.0%
    R2: mean= -0.017731  mean_w= -0.017731  rms_w=0.021926  std_w=0.012898  wrap= 0.0%
    R3: mean= +0.272554  mean_w= +0.272554  rms_w=0.272867  std_w=0.013063  wrap= 0.0%
    energy_w = 0.076573

  a5=2 [UP (u,c,t)], ci=149:
    R0: mean= -0.010873  mean_w= -0.010873  rms_w=0.010874  std_w=0.000108  wrap= 0.0%
    R1: mean= +0.028216  mean_w= +0.028

In [7]:
# A4: ALL UP/DOWN PAIRS — RMS ratios across all generations

print('A4: ALL UP(a5=2)/DOWN(a5=0) PAIRS')
print('=' * 80)
print(f'  {"a7":>3s} {"gen":>4s}  {"ci_d":>5s} {"ci_u":>5s} {"Dci":>5s}  '
      f'{"R0 u/d":>8s} {"R1 u/d":>8s} {"R2 u/d":>8s} {"R3 u/d":>8s}  '
      f'{"E u/d":>8s}  {"R3_d":>8s} {"R3_u":>8s}')
for a7_val in range(6):
    ci_d = sector_to_ci.get((1, 0, a7_val))
    ci_u = sector_to_ci.get((1, 2, a7_val))
    if ci_d is None or ci_u is None: continue
    gen = gen_map[a7_val]
    idx_d, idx_u = ci_d-1, ci_u-1
    ratios = [profiles['rms_w'][idx_u, k] / profiles['rms_w'][idx_d, k]
              if profiles['rms_w'][idx_d, k] > 1e-10 else 0 for k in range(4)]
    E_r = profiles['energy_w'][idx_u] / profiles['energy_w'][idx_d]
    print(f'  {a7_val:3d} {gen:4d}  {ci_d:5d} {ci_u:5d} {ci_u-ci_d:+5d}  '
          f'{ratios[0]:8.4f} {ratios[1]:8.4f} {ratios[2]:8.4f} {ratios[3]:8.4f}  '
          f'{E_r:8.4f}  '
          f'{profiles["rms_w"][idx_d,3]:8.4f} {profiles["rms_w"][idx_u,3]:8.4f}')

# Also check: do the mean_w signs differ between up and down?
print(f'\n  SIGN PATTERN of mean_w(R3) at up vs down:')
for a7_val in range(6):
    ci_d = sector_to_ci.get((1, 0, a7_val))
    ci_u = sector_to_ci.get((1, 2, a7_val))
    if ci_d is None or ci_u is None: continue
    gen = gen_map[a7_val]
    sign_d = '+' if profiles['mean_w'][ci_d-1, 3] > 0 else '-'
    sign_u = '+' if profiles['mean_w'][ci_u-1, 3] > 0 else '-'
    print(f'  a7={a7_val} gen{gen}: DOWN({sign_d}) UP({sign_u})  '
          f'same={sign_d==sign_u}')

A4: ALL UP(a5=2)/DOWN(a5=0) PAIRS
   a7  gen   ci_d  ci_u   Dci    R0 u/d   R1 u/d   R2 u/d   R3 u/d     E u/d      R3_d     R3_u
    0    1     71    29   -42   22.3782  10.3181  10.4485   3.1789   25.8525    0.5858   1.8623
    1    2    101    59   -42    8.0239   6.2273  22.5746   1.0990    3.6289    0.3308   0.3636
    2    3    191   149   -42    0.9908   1.0264   0.9559   1.0717    1.1418    0.2795   0.2995
    3    1     41   209  +168    0.0430   0.0355   0.0334   0.1456    0.0141    2.0761   0.3023
    4    2     11   179  +168    0.0053   0.0170   0.0250   0.1635    0.0070    1.8465   0.3019
    5    3    131    89   -42    0.7497   2.2077   1.2675   0.7231    0.5867    0.2880   0.2083

  SIGN PATTERN of mean_w(R3) at up vs down:
  a7=0 gen1: DOWN(+) UP(-)  same=False
  a7=1 gen2: DOWN(+) UP(+)  same=True
  a7=2 gen3: DOWN(+) UP(-)  same=False
  a7=3 gen1: DOWN(+) UP(-)  same=False
  a7=4 gen2: DOWN(-) UP(-)  same=True
  a7=5 gen3: DOWN(+) UP(-)  same=False

In [8]:
# A5: BRANCH DECOMPOSITION — what j index drives R_k at t and b crossings?
# A6: CROSS-LEVEL CORRELATIONS — how do levels talk to each other?

print('A5: BRANCH DECOMPOSITION AT TOP (ci=149) AND BOTTOM (ci=191)')
print('=' * 80)

for label, ci in [('TOP', 149), ('BOT', 191), ('WRAPPING', 11), ('LEPTON g1', 31)]:
    ci_idx = ci - 1
    print(f'\n  {label} (ci={ci}):')
    
    for k in range(4):
        Rk_w = wrap(np.array([res[br][ci_idx, k] for br in branches]))
        var_total = np.var(Rk_w)
        if var_total < 1e-15:
            print(f'    R{k}: zero variance (all branches identical)')
            continue
        
        for j_level in range(4):
            j_this = j_vals[:, j_level]
            p_this = primes[j_level]
            group_means = [np.mean(Rk_w[j_this == j]) for j in range(p_this)]
            var_between = np.var(group_means)
            eta2 = var_between / var_total
            if eta2 > 0.005:
                means_str = ','.join(f'{m:+.5f}' for m in group_means)
                print(f'    R{k} by j{j_level}(p={p_this}): eta2={eta2:.4f} [{means_str}]')

print(f'\n\nA6: CROSS-LEVEL CORRELATIONS')
print('=' * 80)

for label, ci in [('TOP', 149), ('BOT', 191), ('WRAPPING', 11)]:
    ci_idx = ci - 1
    Rk_all = np.column_stack([
        wrap(np.array([res[br][ci_idx, k] for br in branches]))
        for k in range(4)
    ])
    corr = np.corrcoef(Rk_all.T)
    print(f'\n  {label} (ci={ci}) correlation matrix (Pearson r):')
    print(f'         R0       R1       R2       R3')
    for i in range(4):
        print(f'    R{i}  [{" ".join(f"{corr[i,j]:+7.4f}" for j in range(4))}]')

A5: BRANCH DECOMPOSITION AT TOP (ci=149) AND BOTTOM (ci=191)

  TOP (ci=149):
    R0 by j0(p=2): eta2=1.0000 [-0.01098,-0.01077]
    R1 by j0(p=2): eta2=0.9081 [+0.02766,+0.02877]
    R1 by j1(p=3): eta2=0.0919 [+0.02800,+0.02822,+0.02843]
    R2 by j0(p=2): eta2=0.6595 [-0.04264,-0.04077]
    R2 by j1(p=3): eta2=0.2705 [-0.04243,-0.04170,-0.04097]
    R2 by j2(p=5): eta2=0.0700 [-0.04213,-0.04192,-0.04170,-0.04149,-0.04127]
    R3 by j0(p=2): eta2=0.3036 [-0.30016,-0.29888]
    R3 by j1(p=3): eta2=0.2797 [-0.30027,-0.29952,-0.29876]
    R3 by j2(p=5): eta2=0.2794 [-0.30037,-0.29995,-0.29953,-0.29910,-0.29863]
    R3 by j3(p=7): eta2=0.1371 [-0.30016,-0.29995,-0.29973,-0.29952,-0.29930,-0.29909,-0.29887]

  BOT (ci=191):
    R0 by j0(p=2): eta2=1.0000 [-0.01098,-0.01097]
    R1 by j0(p=2): eta2=0.9420 [+0.02746,+0.02754]
    R1 by j1(p=3): eta2=0.0580 [+0.02749,+0.02750,+0.02751]
    R2 by j0(p=2): eta2=0.7766 [-0.04373,-0.04356]
    R2 by j1(p=3): eta2=0.1931 [-0.04370,-0.04364,-0.043

In [9]:
# A7: ENERGY PROFILE + A8: NON-COPRIME + A9: LEPTONS + A10: H3 FILTER + A12: WRAPPING MAP

# ─── A7: ENERGY ───
print('A7: ENERGY PROFILE')
print('=' * 80)
cop_E = [profiles['energy_w'][ci-1] for ci in cis]
noncop_E = [profiles['energy_w'][i] for i in range(N) if gcd(int(t_eval[i]), P4) > 1]
print(f'  Coprime ({len(cop_E)}):  mean={np.mean(cop_E):.4f} std={np.std(cop_E):.4f} min={np.min(cop_E):.4f} max={np.max(cop_E):.4f}')
print(f'  Non-cop ({len(noncop_E)}): mean={np.mean(noncop_E):.4f} std={np.std(noncop_E):.4f} min={np.min(noncop_E):.4f} max={np.max(noncop_E):.4f}')

# Energy by sector
for a3_val in [0, 1]:
    s = 'LEP' if a3_val == 0 else 'QUARK'
    for a5_val in [0, 2]:
        ch = 'down' if a5_val == 0 else 'up'
        energies = []
        for a7_val in range(6):
            ci = sector_to_ci.get((a3_val, a5_val, a7_val))
            if ci: energies.append(profiles['energy_w'][ci-1])
        if energies:
            print(f'  {s} {ch}: mean E = {np.mean(energies):.6f} ({len(energies)} crossings)')

# ─── A8: NON-COPRIME ───
print(f'\n{"=" * 80}')
print('A8: NON-COPRIME CROSSINGS')
print('=' * 80)
for p in primes:
    mults = [ci for ci in range(p, P4+1, p) if gcd(ci, P4) == p]  # divisible by EXACTLY p
    if mults:
        E_vals = [profiles['energy_w'][ci-1] for ci in mults]
        R3_vals = [profiles['rms_w'][ci-1, 3] for ci in mults]
        print(f'  div by {p} only ({len(mults)}): mean E={np.mean(E_vals):.4f}, mean R3={np.mean(R3_vals):.4f}')

# Also: ci divisible by multiple primes
for combo in [(2,3), (2,5), (2,7), (3,5), (3,7), (5,7), (2,3,5), (2,3,7), (2,5,7), (3,5,7)]:
    prod_c = prod(combo)
    mults = [ci for ci in range(prod_c, P4+1, prod_c) if ci <= P4]
    if mults:
        E_vals = [profiles['energy_w'][ci-1] for ci in mults]
        R3_vals = [profiles['rms_w'][ci-1, 3] for ci in mults]
        label = '*'.join(str(c) for c in combo)
        print(f'  div by {label}={prod_c} ({len(mults)}): mean E={np.mean(E_vals):.4f}, mean R3={np.mean(R3_vals):.4f}')

# ─── A9: LEPTONS ───
print(f'\n{"=" * 80}')
print('A9: LEPTON SECTOR')
print('=' * 80)
for a5_val in [0, 2]:
    ch = 'down' if a5_val == 0 else 'up'
    for a7_val in range(6):
        ci = sector_to_ci.get((0, a5_val, a7_val))
        if ci is None: continue
        gen = gen_map.get(a7_val, '?')
        ci_idx = ci - 1
        wf3 = profiles['wrap_frac'][ci_idx, 3]
        wf_str = f'{wf3*100:.0f}%' if wf3 > 0.005 else '  .'
        print(f'  L {ch} gen{gen} a7={a7_val} ci={ci:3d}: '
              f'R3_rms={profiles["rms_w"][ci_idx,3]:.6f} wrap={wf_str:>4s} '
              f'E={profiles["energy_w"][ci_idx]:.6f}')

# Lepton CP ratios
for cp_name in ['LEPTON']:
    ch_a3 = CP_PAIRS[cp_name][0]
    a7_g1 = CP_PAIRS[cp_name][1]
    a7_g2 = CP_PAIRS[cp_name][2]
    ci_g1 = sector_to_ci.get((ch_a3, 0, a7_g1))
    ci_g2 = sector_to_ci.get((ch_a3, 0, a7_g2))
    if ci_g1 and ci_g2:
        print(f'\n  {cp_name} CP pair: ci={ci_g1}/{ci_g2} Dci={abs(ci_g1-ci_g2)}')
        for k in range(4):
            r = profiles['rms_w'][ci_g1-1, k] / profiles['rms_w'][ci_g2-1, k]
            print(f'    R{k}: CP = {r:.6f}')

# ─── A10: H3^2 FILTER ───
print(f'\n{"=" * 80}')
print('A10: H3^2 FILTER VERIFICATION')
print('=' * 80)
H3_sq_formula = 30**2 / (30**2 + (2*np.pi)**2 * 210)
print(f'  Analytical: H3^2 = {H3_sq_formula:.6f}')

# Measure from far-field cascade: ratio of R3 to R0 steady-state amplitudes
far_cis = [ci for ci in cis if ci > 100]
r3_far = np.mean([profiles['rms_w'][ci-1, 3] for ci in far_cis])
r2_far = np.mean([profiles['rms_w'][ci-1, 2] for ci in far_cis])
r1_far = np.mean([profiles['rms_w'][ci-1, 1] for ci in far_cis])
r0_far = np.mean([profiles['rms_w'][ci-1, 0] for ci in far_cis])
print(f'  Far-field RMS: R0={r0_far:.6f} R1={r1_far:.6f} R2={r2_far:.6f} R3={r3_far:.6f}')
print(f'  R3/R0 = {r3_far/r0_far:.4f}')
print(f'  R3/R1 = {r3_far/r1_far:.4f}')
print(f'  R2/R1 = {r2_far/r1_far:.4f}')
print(f'  R1/R0 = {r1_far/r0_far:.4f}')

# The filter at each level: H_k = steady_state(R_k) / steady_state(driving_at_k)
# R0 driving: epsilon*sin(omega*t). Steady state: R0_ss = epsilon*omega/(omega^2+kappa^2)
# But at integer t, we measure the RESIDUAL, not the amplitude.
# The residual depends on the phase offset.

# Instead, compare ratio of consecutive levels:
print(f'\n  Level ratios (cascade coupling):')
print(f'    R1/R0 × p2 = {r1_far/r0_far * p2:.4f}')
print(f'    R2/R1 × p3 = {r2_far/r1_far * p3:.4f}')
print(f'    R3/R2 × p4 = {r3_far/r2_far * p4:.4f}')

# ─── A12: COMPLETE WRAPPING MAP ───
print(f'\n{"=" * 80}')
print('A12: COMPLETE WRAPPING MAP')
print('=' * 80)
print(f'  {"ci":>4s} {"a3":>3s} {"a5":>3s} {"a7":>3s} {"g":>2s}  '
      f'{"wR0":>5s} {"wR1":>5s} {"wR2":>5s} {"wR3":>5s}  '
      f'{"R3rms":>8s} {"E":>8s}')
for ci_sorted_idx in np.argsort(cis):
    ci = cis[ci_sorted_idx]
    ci_idx = ci - 1
    wf = profiles['wrap_frac'][ci_idx]
    gen = gen_map.get(a7[ci_sorted_idx], '?')
    wf_strs = [f'{w*100:4.0f}%' if w > 0.005 else '   .' for w in wf]
    print(f'  {ci:4d} {a3[ci_sorted_idx]:3d} {a5[ci_sorted_idx]:3d} {a7[ci_sorted_idx]:3d} {gen:2}  '
          f'{wf_strs[0]:>5s} {wf_strs[1]:>5s} {wf_strs[2]:>5s} {wf_strs[3]:>5s}  '
          f'{profiles["rms_w"][ci_idx,3]:8.4f} {profiles["energy_w"][ci_idx]:8.4f}')

print(f'\n{"=" * 80}')
print('ALL COMPUTATIONAL ANALYSES (A1-A10, A12) COMPLETE.')
print('=' * 80)

A7: ENERGY PROFILE
  Coprime (48):  mean=2.3541 std=4.2146 min=0.0073 max=13.3550
  Non-cop (162): mean=2.2968 std=4.1758 min=0.0035 max=13.3025
  LEP down: mean E = 2.335058 (6 crossings)
  LEP up: mean E = 1.979245 (6 crossings)
  QUARK down: mean E = 3.454021 (6 crossings)
  QUARK up: mean E = 1.869761 (6 crossings)

A8: NON-COPRIME CROSSINGS
  div by 2 only (48): mean E=2.2181, mean R3=0.6267
  div by 3 only (24): mean E=2.2124, mean R3=0.5715
  div by 5 only (12): mean E=1.8948, mean R3=0.4507
  div by 7 only (8): mean E=1.8046, mean R3=0.5561
  div by 2*3=6 (35): mean E=2.2019, mean R3=0.6035
  div by 2*5=10 (21): mean E=2.2073, mean R3=0.5864
  div by 2*7=14 (15): mean E=2.0369, mean R3=0.5475
  div by 3*5=15 (14): mean E=2.0487, mean R3=0.6253
  div by 3*7=21 (10): mean E=1.8476, mean R3=0.5621
  div by 5*7=35 (6): mean E=1.6621, mean R3=0.5980
  div by 2*3*5=30 (7): mean E=1.5816, mean R3=0.5115
  div by 2*3*7=42 (5): mean E=1.2479, mean R3=0.5174
  div by 2*5*7=70 (3): mean E